In [14]:
import chess
import chess.engine
import random
from collections import defaultdict
from sarfa_saliency import computeSaliencyUsingSarfa

In [15]:
def q_values(board, legal_moves_original_state, original_optimal_move, multipv=3, runtime=2.0):
    current_legal_moves = set(board.legal_moves)
    original_legal_moves = set(legal_moves_original_state)
    valid_moves = current_legal_moves.intersection(original_legal_moves)

    options = chess_engine.analyse(board, chess.engine.Limit(time=runtime), multipv=multipv)
    original_optimal_move_non_string = None
    if (original_optimal_move is None):
        original_optimal_move_non_string = options[0]["pv"][0]
        original_optimal_move = str(options[0]["pv"][0])
    
    score_per_move = defaultdict(int)

    for option in options:
        score = 0
        white_score = str(option["score"].black())
        move = str(option["pv"][0])
        if (white_score[0] == "#"):
            # mate
            if (white_score[1] == "+"):
                score = 40
            else:
                score = -40
        else:
            score = round(option["score"].black().cp/100.0, 2)
        score_per_move[move] = score

    q_vals = {}
    for valid_move in valid_moves:
        q_vals[str(valid_move)] = score_per_move[str(valid_move)]
    
    return q_vals, original_optimal_move, original_optimal_move_non_string

In [28]:
import chess
import chess.engine
from sys import platform as _platform
import random

def get_engine(engine_file = './stockfish_15_x64_avx2'):
    chess_engine = chess.engine.SimpleEngine.popen_uci(engine_file)
    return chess_engine

max_actions_considered = 100
max_pieces_removed_concurrently = 3

# FEN = "k7/1b6/8/8/8/8/6B1/7K w - - 0 1"
# FEN = "rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq - 0 1"
importance_values = {
        'a1' : {'pos': chess.A1, 'saliency': -2},
        'a2' : {'pos': chess.A2, 'saliency': -2},
        'a3' : {'pos': chess.A3, 'saliency': -2},
        'a4' : {'pos': chess.A4, 'saliency': -2},
        'a5' : {'pos': chess.A5, 'saliency': -2},
        'a6' : {'pos': chess.A6, 'saliency': -2},
        'a7' : {'pos': chess.A7, 'saliency': -2},
        'a8' : {'pos': chess.A8, 'saliency': -2},
        'b1' : {'pos': chess.B1, 'saliency': -2},
        'b2' : {'pos': chess.B2, 'saliency': -2},
        'b3' : {'pos': chess.B3, 'saliency': -2},
        'b4' : {'pos': chess.B4, 'saliency': -2},
        'b5' : {'pos': chess.B5, 'saliency': -2},
        'b6' : {'pos': chess.B6, 'saliency': -2},
        'b7' : {'pos': chess.B7, 'saliency': -2},
        'b8' : {'pos': chess.B8, 'saliency': -2},
        'c1' : {'pos': chess.C1, 'saliency': -2},
        'c2' : {'pos': chess.C2, 'saliency': -2},
        'c3' : {'pos': chess.C3, 'saliency': -2},
        'c4' : {'pos': chess.C4, 'saliency': -2},
        'c5' : {'pos': chess.C5, 'saliency': -2},
        'c6' : {'pos': chess.C6, 'saliency': -2},
        'c7' : {'pos': chess.C7, 'saliency': -2},
        'c8' : {'pos': chess.C8, 'saliency': -2},
        'd1' : {'pos': chess.D1, 'saliency': -2},
        'd2' : {'pos': chess.D2, 'saliency': -2},
        'd3' : {'pos': chess.D3, 'saliency': -2},
        'd4' : {'pos': chess.D4, 'saliency': -2},
        'd5' : {'pos': chess.D5, 'saliency': -2},
        'd6' : {'pos': chess.D6, 'saliency': -2},
        'd7' : {'pos': chess.D7, 'saliency': -2},
        'd8' : {'pos': chess.D8, 'saliency': -2},
        'e1' : {'pos': chess.E1, 'saliency': -2},
        'e2' : {'pos': chess.E2, 'saliency': -2},
        'e3' : {'pos': chess.E3, 'saliency': -2},
        'e4' : {'pos': chess.E4, 'saliency': -2},
        'e5' : {'pos': chess.E5, 'saliency': -2},
        'e6' : {'pos': chess.E6, 'saliency': -2},
        'e7' : {'pos': chess.E7, 'saliency': -2},
        'e8' : {'pos': chess.E8, 'saliency': -2},
        'f1' : {'pos': chess.F1, 'saliency': -2},
        'f2' : {'pos': chess.F2, 'saliency': -2},
        'f3' : {'pos': chess.F3, 'saliency': -2},
        'f4' : {'pos': chess.F4, 'saliency': -2},
        'f5' : {'pos': chess.F5, 'saliency': -2},
        'f6' : {'pos': chess.F6, 'saliency': -2},
        'f7' : {'pos': chess.F7, 'saliency': -2},
        'f8' : {'pos': chess.F8, 'saliency': -2},
        'g1' : {'pos': chess.G1, 'saliency': -2},
        'g2' : {'pos': chess.G2, 'saliency': -2},
        'g3' : {'pos': chess.G3, 'saliency': -2},
        'g4' : {'pos': chess.G4, 'saliency': -2},
        'g5' : {'pos': chess.G5, 'saliency': -2},
        'g6' : {'pos': chess.G6, 'saliency': -2},
        'g7' : {'pos': chess.G7, 'saliency': -2},
        'g8' : {'pos': chess.G8, 'saliency': -2},
        'h1' : {'pos': chess.H1, 'saliency': -2},
        'h2' : {'pos': chess.H2, 'saliency': -2},
        'h3' : {'pos': chess.H3, 'saliency': -2},
        'h4' : {'pos': chess.H4, 'saliency': -2},
        'h5' : {'pos': chess.H5, 'saliency': -2},
        'h6' : {'pos': chess.H6, 'saliency': -2},
        'h7' : {'pos': chess.H7, 'saliency': -2},
        'h8' : {'pos': chess.H8, 'saliency': -2}}
# board_positions = ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'd1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'h8']
# board_positions = [board_position for board_position in board_positions if board.piece_at(importance_values[board_position]['pos'])]
runtime = 5.0
chess_engine = get_engine()

FEN1 = '3r1rk1/1b2bpp1/p3p2p/R3N3/2pB1P2/2P3qP/PP1Q2P1/5RK1 b - - 0 1'
board = chess.Board(FEN1)

legal_moves_original_state = list(board.legal_moves)
q_vals_before_perturb, original_optimal_move, original_optimal_move_non_string = q_values(board, legal_moves_original_state, None, multipv=100, runtime=runtime)

FEN2 = '3r1rk1/4bpp1/p3p2p/R3N3/2pB1P2/2P3qP/PP1Q2P1/5RK1 b - - 0 1'
board_2 = chess.Board(FEN2)
q_vals_after_perturb, original_optimal_move, original_optimal_move_non_string = q_values(board_2, legal_moves_original_state, original_optimal_move, multipv=100, runtime=runtime)

saliency, dP, K, QmaxAnswer, action_gap_before_perturbation, action_gap_after_perturbation = computeSaliencyUsingSarfa(original_optimal_move, q_vals_before_perturb, q_vals_after_perturb)
original_saliency = saliency
print(f"original saliency: {original_saliency}")

board_positions = ['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'd1', 'd2', 'd3', 'd4', 'd5', 'd6', 'd7', 'd8', 'e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7', 'e8', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'g1', 'g2', 'g3', 'g4', 'g5', 'g6', 'g7', 'g8', 'h1', 'h2', 'h3', 'h4', 'h5', 'h6', 'h7', 'h8']
board_positions = [board_position for board_position in board_positions if board_2.piece_at(importance_values[board_position]['pos'])]

for removed_position in board_positions:
    print(removed_position)
    if (board_2.piece_at(importance_values[removed_position]['pos']) == chess.Piece(chess.KING, chess.WHITE) or board_2.piece_at(importance_values[removed_position]['pos']) == chess.Piece(chess.KING, chess.BLACK)):
        continue
    piece_removed = board_2.remove_piece_at(importance_values[removed_position]['pos'])

    try:
        valid = True
        if (piece_removed is None):
            print(piece_removed)
            print("Not valid ...")
            valid = False
        elif (piece_removed == chess.Piece(chess.KING, chess.WHITE) or piece_removed == chess.Piece(chess.KING, chess.BLACK)) or board_2.was_into_check():
            # make sure that you don't get into a check from removing the piece
            print(piece_removed)
            print("King or Queen ...")
            valid = False
        if (not valid):
            board_2.set_piece_at(importance_values[removed_position]['pos'], piece_removed)
            continue
        q_vals_after_perturb, original_optimal_move, original_optimal_move_non_string = q_values(board_2, legal_moves_original_state, original_optimal_move, multipv=100, runtime=runtime)
        saliency, dP, K, QmaxAnswer, action_gap_before_perturbation, action_gap_after_perturbation = computeSaliencyUsingSarfa(original_optimal_move, q_vals_before_perturb, q_vals_after_perturb)
        print(f'removed position: {removed_position} - saliency: {saliency} - delta: {saliency - original_saliency}')
    except Exception as e:
        print(e)
        print()
        board_2.set_piece_at(importance_values[removed_position]['pos'], piece_removed)
        continue
    
    
    board_2.set_piece_at(importance_values[removed_position]['pos'], piece_removed)

# if (valid):
#     print(old_board)
#     print()
#     print(board)





# FEN = "rnbqkbnr/pppppppp/8/8/3P4/8/PPP1PPPP/RNBQKBNR b KQkq - 0 1"
# # FEN = "1k6/8/K7/8/8/2Q5/8/8 w - - 0 1"

# board = chess.Board(FEN)

# # board = chess.Board()
# print(board)

# info = chess_engine.analyse(board, chess.engine.Limit(time=2.0), multipv=3)  # 2-second thinking time limit
# for entry in info:
#     x = str(entry["score"].white())
#     print(x)
#     print()
#     print()

original saliency: 0.7434105214991804
a2
removed position: a2 - saliency: 0.7153894477519174 - delta: -0.028021073747263037
a5
removed position: a5 - saliency: 0.4238055314195221 - delta: -0.3196049900796583
a6
removed position: a6 - saliency: 0.742997676662241 - delta: -0.00041284483693937446
b2
removed position: b2 - saliency: 0.6440918373903047 - delta: -0.09931868410887568
c3
removed position: c3 - saliency: 0.723845287986544 - delta: -0.019565233512636393
c4
removed position: c4 - saliency: 0.717471315497586 - delta: -0.025939206001594384
d2
removed position: d2 - saliency: 0.8434267281651419 - delta: 0.10001620666596145
d4
removed position: d4 - saliency: 0.22329277015225377 - delta: -0.5201177513469266
d8
'd8d4'

e5
removed position: e5 - saliency: 0.5776617382855435 - delta: -0.16574878321363695
e6
removed position: e6 - saliency: 0.7484862057049066 - delta: 0.005075684205726239
e7
removed position: e7 - saliency: 0.8684568388503257 - delta: 0.12504631735114524
f1
removed posit